# Mapping ARCHS4 Tissue Categories

The objective of this notebook is to resolve the tissue annotations of the ARCHS4 data using UBERON IDs, which are also used by single-cell collection CellxGene to map to tissue categories. This would allow us to establish a direct link between ARCHS4 metadata fields and CellxGene tissue categories through UBERON IDs.

In [1]:
#Imports
import scanpy as sc
import anndata as ad
import pandas as pd
import numpy as np
from pathlib import Path
import os
import json
import re
import matplotlib.pyplot as plt
import scipy.sparse as sp
from pathlib import Path
import re
import h5py

## Load Data

In [2]:
# Original ARCHS4
BULK_DIR = Path("/cluster/work/boeva/rquiles/data/ARCHS4-data/data/processed/archs4")
EXPRESSION_H5 = BULK_DIR / "expression.h5"
GENE_LIST_TXT = BULK_DIR / "gene_list.txt"
METADATA_CSV = BULK_DIR / "metadata.csv"

metadata = pd.read_csv(METADATA_CSV)

In [3]:
metadata.head(3)

,sample_id,geo_accession,series_id,platform_id,instrument,organism,characteristics,extract_protocol,source_name,title,taxid,type,total_counts,n_detected_genes
0,GSM1000981,GSM1000981,GSE29282,GPL15433,Illumina HiSeq 1000,Homo sapiens,"treatment: siNT,cell line: OCI-LY1",ChIP-seq libraries were prepared using the Ill...,Human DLBCL cel line,OCI-LY1_48hrs_mRNAseq_3x_siNT_R1,9606,SRA,70707856.0,32704
1,GSM1000982,GSM1000982,GSE29282,GPL15433,Illumina HiSeq 1000,Homo sapiens,"treatment: siNT,cell line: OCI-LY1",ChIP-seq libraries were prepared using the Ill...,Human DLBCL cel line,OCI-LY1_48hrs_mRNAseq_3x_siNT_R2,9606,SRA,58297056.0,32070
2,GSM1000983,GSM1000983,GSE29282,GPL15433,Illumina HiSeq 1000,Homo sapiens,"treatment: siNT,cell line: OCI-LY1",ChIP-seq libraries were prepared using the Ill...,Human DLBCL cel line,OCI-LY1_48hrs_mRNAseq_3x_siNT_R3,9606,SRA,65223632.0,32713


In [4]:
# CellxGene Data
CELLXGENE_DIR = Path("/cluster/work/boeva/atheus/data_cellxgene")
FILE = CELLXGENE_DIR / "partition_100.h5ad"

adata_sc = sc.read_h5ad(FILE, backed="r")

In [5]:
adata_sc.obs.head(3)

,soma_joinid,dataset_id,assay,assay_ontology_term_id,cell_type,cell_type_ontology_term_id,development_stage,development_stage_ontology_term_id,disease,disease_ontology_term_id,...,tissue_ontology_term_id,tissue_type,tissue_general,tissue_general_ontology_term_id,raw_sum,nnz,raw_mean_nnz,raw_variance_nnz,n_measured_vars,n_counts
0,35111238,4a5b00e0-1ba3-4fd4-af89-d3512eb20720,10x 3' v3,EFO:0009922,hematopoietic precursor cell,CL:0008001,41-year-old stage,HsapDv:0000135,normal,PATO:0000461,...,UBERON:0000178,tissue,blood,UBERON:0000178,1580.0,940,1.680851,3.906555,28560,1580.0
1,35111239,4a5b00e0-1ba3-4fd4-af89-d3512eb20720,10x 3' v3,EFO:0009922,hematopoietic precursor cell,CL:0008001,30-year-old stage,HsapDv:0000124,normal,PATO:0000461,...,UBERON:0000178,tissue,blood,UBERON:0000178,2504.0,1121,2.233720,16.316755,28560,2504.0
2,35111240,4a5b00e0-1ba3-4fd4-af89-d3512eb20720,10x 3' v3,EFO:0009922,hematopoietic precursor cell,CL:0008001,36-year-old stage,HsapDv:0000130,normal,PATO:0000461,...,UBERON:0000178,tissue,blood,UBERON:0000178,1771.0,886,1.998871,8.496044,28560,1771.0


## ARHS4 metadata --> UBERON IDs

Steps:
1) Find key:value pairs in characteristic fields, determine curation queue.
2) Exact matching to UBERON labels and synonims
3) Fuzzy matching

### 1) Find key:value pairs and determine curation queue

First, we split the characteristics field to generate new fields denoted by the key:value pairs found inside that field. We then assess the coverage that these fields provide.

In [6]:
# characteristics are joined by "Xx-xX" or by a comma followed by a new "key:"
SPLIT = re.compile(r"Xx-xX|,(?=\s*[^,:]{1,60}:)")

# explicit key lists, in priority order (first key a sample has wins)
TISSUE_KEYS = [
    "tissue", "tissue type", "tissuetype", "organ", "source organ", "organ site", "source tissue",
    "tissue source", "tissue origin", "tissue of origin", "tissue (isolated from)", "body site",
    "anatomical site", "anatomic region", "sampling site", "biopsy site", "biopsy location",
    "organ/tissue", "tissue region", "tissue subtype",
    "tissue/celltype", "tissue/cell type", "tissue/cell type source",
    "cell line origin",                     # tissue a cell line came from; these samples are still flagged has_cell_line
]
CELL_TYPE_KEYS = [
    "cell type", "celltype", "curated-cell-type", "cell type sorted", "sample cell type", "source cell type",
    "endocrine cell-type", "inferred cell type", "cell type or stage", "cell population", "cell subtype",
    "cell types", "cell line/type",
]
CELL_LINE_KEYS = [
    "cell line", "cell line name", "cellline", "cell line id", "inhouse cell line name",
    "source cell line", "originating cell line", "parental cell line", "cell line background",
]
FALLBACK_TISSUE_KEYS = ["organ system"]     # too coarse for UBERON; only used when nothing else is available

# reviewed and deliberately ignored; hidden from the "consider adding" printout
SKIP_KEYS = {
    "organism", "organisms", "organ donation death type", "organoidid", "tissue preparation",
    "cell type surface markers", "age at biopsy", "tissuestoragetime", "tissue collection",
    "tissue composition", "tissue disease state", "cancer cell line composition",
    "cell line group", "sample cell type cat",
}
SUGGEST = re.compile(r"tissue|organ|body.?site|anatom|biopsy|cell.?type|cell.?line", re.I)   # only to suggest extra keys (case-insensitive)

NA = {"", "nan", "none", "na", "n/a", "not applicable", "missing", "unknown", "not available"}

# String normalizer function
def norm(s) -> str:
    s = re.sub(r"\s+", " ", re.sub(r"_+", " ", str(s).lower())).strip(" ,;")
    return "" if s in NA else s

# ---------------- 1) load ----------------
metadata = pd.read_csv(METADATA_CSV, dtype=str, low_memory=False)
meta = metadata.set_index("sample_id")
meta.index = meta.index.astype(str)
assert meta.index.is_unique, "duplicated sample_id in metadata.csv"
print(f"{len(meta):,} samples")

# ---------------- 2) characteristics -> key: value pairs ----------------
rows = []
for sid, txt in meta["characteristics"].dropna().items():
    for part in SPLIT.split(txt):
        if ":" in part:
            k, v = part.split(":", 1)
            k, v = norm(k), norm(v)
            if k and v:
                rows.append((sid, k, v))

# stores the key-value pairs associated to the samples that had them
kv = pd.DataFrame(rows, columns=["sample_id", "key", "value"]).drop_duplicates(["sample_id", "key"])
print(f"{len(kv):,} key: value pairs")

# Number of samples for each given key, sorted
key_counts = kv.groupby("key")["sample_id"].nunique().sort_values(ascending=False)
print("\nmost common keys:\n", key_counts.head(30).to_string())

listed = set(TISSUE_KEYS + CELL_TYPE_KEYS + CELL_LINE_KEYS + FALLBACK_TISSUE_KEYS) | SKIP_KEYS
suggest = key_counts[[k for k in key_counts.index if SUGGEST.search(k) and k not in listed]]
print("\nkeys NOT in the lists that look relevant (consider adding):\n", suggest.head(25).to_string())

def pick(keys):
    """Value (and key) of the first key in `keys` that each sample has (e.g. only one tissue key per sample is recovered)"""
    rank = {k: i for i, k in enumerate(keys)}
    sub = kv[kv["key"].isin(rank)].assign(r=lambda d: d["key"].map(rank)).sort_values(["sample_id", "r"])
    sub = sub.drop_duplicates("sample_id").set_index("sample_id") # mantains only values from top keys, as they are sorted
    return sub["value"], sub["key"]

tissue_v, tissue_k = pick(TISSUE_KEYS)
celltype_v, _      = pick(CELL_TYPE_KEYS)
cellline_v, _      = pick(CELL_LINE_KEYS)
fallback_v, _      = pick(FALLBACK_TISSUE_KEYS)

# ---------------- 3) per-sample table ----------------
# all dataframes sharing the index ensures the mapping correct throughout
samples = pd.DataFrame(index=meta.index)
samples.index.name = "sample_id"
samples["series_id"]       = meta["series_id"]
samples["source_name"]     = meta["source_name"].map(norm)
samples["tissue_key"]      = tissue_k
samples["tissue_value"]    = tissue_v
samples["cell_type_value"] = celltype_v
samples["cell_line_value"] = cellline_v
samples["tissue_fallback"] = fallback_v      # e.g. "nervous system"; use only if tissue_value and source_name fail
samples["title"]           = meta["title"].map(norm)
samples["characteristics"] = meta["characteristics"].map(norm)
samples = samples.replace("", np.nan)
samples["has_cell_line"]   = samples["cell_line_value"].notna()

print("\nfield presence:")
for c in ["source_name", "tissue_value", "cell_type_value", "cell_line_value", "tissue_fallback"]:
    print(f"  {c:<16} {samples[c].notna().mean():6.1%}")
print(f"  no tissue value and no source name: {(samples['tissue_value'].isna() & samples['source_name'].isna()).sum():,}")
print("  tissue_value by key:\n" + samples["tissue_key"].value_counts().to_string())

# ---------------- 4) unique strings ranked by coverage ----------------
# Table of unique strings of a given column ordered by the amount of samples where they appear
def ranked(col):
    s = samples[col].dropna() # Series whose values are strings 
    g = pd.DataFrame({
        "n_samples": s.groupby(s).size(), # group using the unique values in s
        "n_series": samples.loc[s.index, "series_id"].groupby(s).nunique(),
        "example_id": s.groupby(s).apply(lambda x: x.index[0]),
        "frac_cell_line": samples.loc[s.index, "has_cell_line"].groupby(s).mean().round(2),
    }).sort_values("n_samples", ascending=False)
    g["cum_frac_samples"] = g["n_samples"].cumsum() / len(samples)
    g.index.name = "string"
    return g

queues = {}
for col in ["source_name", "tissue_value", "cell_type_value"]:
    queues[col] = r = ranked(col)
    cov = ", ".join(f"top {n}: {r['cum_frac_samples'].iloc[min(n, len(r)) - 1]:.1%}" for n in (100, 500, 1000, 5000))
    print(f"\n{col}: {len(r):,} unique strings | {cov}")
    print(r.head(10).to_string())

# ---------------- 5) combined queue: tissue value if present, else source name ----------------
prim = samples["tissue_value"].fillna(samples["source_name"])
field = pd.Series(np.where(samples["tissue_value"].notna(), "tissue", "source_name"), index=samples.index)
queue = (pd.DataFrame({"string": prim, "field": field, "series_id": samples["series_id"],
                       "cell_line": samples["has_cell_line"]})
         .dropna(subset=["string"])
         .groupby(["field", "string"])
         .agg(n_samples=("series_id", "size"), n_series=("series_id", "nunique"),
              frac_cell_line=("cell_line", "mean"))
         .sort_values("n_samples", ascending=False)
         .reset_index())
queue["frac_cell_line"] = queue["frac_cell_line"].round(2)
queue["cum_frac_samples"] = queue["n_samples"].cumsum() / len(samples)
queues["combined"] = queue

print(f"\ncombined queue: {len(queue):,} (field, string) pairs")
for n in (200, 500, 1000, 2000, 5000):
    print(f"  top {n:>5}: {queue['cum_frac_samples'].iloc[min(n, len(queue)) - 1]:.1%} of samples")
non_cl = queue[queue["frac_cell_line"] < 0.5]
n_non_cl = int((~samples["has_cell_line"]).sum())
print(f"  excluding cell-line-dominated strings, top 1000 cover "
      f"{non_cl['n_samples'].head(1000).sum() / n_non_cl:.1%} of the {n_non_cl:,} non-cell-line samples")
queue.head(20)

719,432 samples
3,171,089 key: value pairs

most common keys:
 key
cell type             316973
tissue                253350
cell line             205047
treatment             201709
age                    91498
sex                    82933
genotype               80251
gender                 45634
time                   44371
disease state          39295
patient id             27141
batch                  24729
well                   23799
donor                  22631
individual             22453
disease                22291
group                  20780
condition              19937
diagnosis              18634
time point             18118
sample type            17516
timepoint              16422
donor id               16267
subject id             15710
plate                  14829
replicate              14094
genotype/variation     13722
race                   12095
passage                11135
ethnicity              10721

keys NOT in the lists that look relevant (consider adding):
 k

,field,string,n_samples,n_series,frac_cell_line,cum_frac_samples
0,tissue,blood,22841,482,0.09,0.031749
1,tissue,whole blood,15763,172,0.02,0.053659
2,tissue,skin,9438,260,0.23,0.066778
3,source_name,lcl-derived ipsc,8682,3,0.00,0.078846
4,tissue,peripheral blood,8507,374,0.08,0.090670
5,tissue,bone marrow,7936,242,0.09,0.101701
6,source_name,mcf7 cells,7296,26,1.00,0.111842
7,tissue,tumor,6686,117,0.35,0.121136
8,source_name,pbmc,6448,152,0.05,0.130098
9,tissue,pbmc,6319,126,0.00,0.138882


### 2) Exact Matching

Matches strings determined before to the UBERON ontology in the following steps (the first step that fires decides the result, and order reflects confidence):

1) **Clean the text**: fix encoding errors, strip accents, correct known typos.
2) **Generate variants**: strip prefixes/suffixes, swap adjectives for nouns, drop plurals, add “tissue”/”organ”, remove parentheses and tumor words.
3) **Manual decisions**: if any variant is in the hand-curated MANUAL table, use it.
4) **Tumor names**: if a whole variant is a known tumor name, map it to its organ.
5) **Exclusions**: return no ID, with the reason, for non-tissues, control RNA, xenografts, cell line names and in vitro systems.
6) **Special rules**: hematopoietic, nasal epithelium, nasopharynx and blood patterns (blood only if no other body site is mentioned).
7) **Tumor abbreviations**: if any single word is a known tumor abbreviation, map it to its organ.
8) **Ontology lookup**: try official labels and EXACT synonyms on all variants, then RELATED synonyms. If a string matches several terms, report “ambiguous”. Tags mark variant matches (+variant) and tumor context (+tumor).
9) **Organ words**: look for organ keywords. One distinct organ gives a match; several give “ambiguous organ words”.
10) **No match**: otherwise report “no match”.

In [7]:
# obonet library for ontology parsing
import re, obonet
from collections import defaultdict
import networkx as nx

UBERON_OBO = "/cluster/work/boeva/rquiles/ontologies/uberon-basic.obo"
N_REVIEW = 2000          # top strings reviewed by hand

In [8]:
# Parse UBERON ontology
g = obonet.read_obo(UBERON_OBO)                 # obsolete terms are skipped by default
SYN = re.compile(r'^"(.*)"\s+(EXACT|RELATED|NARROW|BROAD)\b')   # obonet keeps synonyms as raw OBO strings
DEF = re.compile(r'^"(.*)"')
_ANC_GRAPH = {}

# Term dictionary of UBERON IDs and label names
terms = {n: {"name": d["name"]} for n, d in g.nodes(data=True)
         if n.startswith("UBERON:") and d.get("name")}

# Dictionary of 3 dictionaries of sets (everything lowecase to ease matching):
# label: name --> {UBERON_IDs}
# exact: synonym --> {UBERON_IDs}
# related : synonym --> {UBERON_IDs}
lookup = {"label": defaultdict(set), "exact": defaultdict(set), "related": defaultdict(set)}
for n, d in g.nodes(data=True):
    if n not in terms:
        continue
    lookup["label"][d["name"].lower()].add(n)
    for s in d.get("synonym", []):
        m = SYN.match(s)
        if m and m.group(2) in ("EXACT", "RELATED"):
            lookup[m.group(2).lower()][m.group(1).lower()].add(n)
print(f"{len(terms):,} UBERON terms")

15,045 UBERON terms


In [9]:
def term_info(g, tid, relations=("is_a", "part_of"), max_ancestors=25):
    """Print what UBERON says about one term. In obonet, edges point child -> parent, keyed by relation."""
    if tid not in g:
        print(f"{tid} not found (obsolete or wrong id)"); return
    name = lambda t: g.nodes[t].get("name", "?") if t in g else "?"
    d = g.nodes[tid]
    print(f"{tid}  {d.get('name')}")

    m = DEF.match(d.get("def", ""))
    if m:
        print(f"  definition: {m.group(1)}")

    syns = [SYN.match(s).groups() for s in d.get("synonym", []) if SYN.match(s)]
    for scope in ("EXACT", "RELATED", "NARROW", "BROAD"):
        vals = [s for s, sc in syns if sc == scope]
        if vals:
            print(f"  {scope.lower()} synonyms: {', '.join(vals)}")

    print("  parents:")
    for _, parent, rel in g.out_edges(tid, keys=True):
        print(f"    {rel:<10} {parent}  {name(parent)}")

    children = [(c, rel) for c, _, rel in g.in_edges(tid, keys=True)]
    print(f"  children: {len(children)}"
          + (": " + ", ".join(f"{name(c)} ({rel})" for c, rel in children[:10]) if children else ""))

    # ancestors: follow only the chosen relations (graph built once per relation set)
    if relations not in _ANC_GRAPH:
        _ANC_GRAPH[relations] = nx.MultiDiGraph((u, v, k) for u, v, k in g.edges(keys=True) if k in relations)
    sub = _ANC_GRAPH[relations]
    anc = nx.descendants(sub, tid) if tid in sub else set()      # child -> parent edges, so descendants = ancestors
    dist = nx.single_source_shortest_path_length(sub, tid)
    print(f"  ancestors via {'/'.join(relations)}: {len(anc)} (closest first)")
    for a in sorted(anc, key=dist.get)[:max_ancestors]:
        print(f"    {a}  {name(a)}")

In [10]:
term_info(g, "UBERON:0001332")

UBERON:0001332  prepuce of penis
  definition: A retractable double-layered fold of skin and mucous membrane that covers the glans penis and protects the urinary meatus when the penis is not erect.
  exact synonyms: penile prepuce, prepuce of male, preputium penis
  related synonyms: foreskin, praeputium, prepucium
  broad synonyms: prepuce, preputium
  parents:
    is_a       UBERON:0011374  prepuce
    part_of    UBERON:0000989  penis
    transformation_of UBERON:0035005  preputial swelling of male
  children: 5: skin of prepuce of penis (part_of), male prepuce epithelium (part_of), male preputial gland (part_of), pearly penile papule (part_of), preputial space of male (part_of)
  ancestors via is_a/part_of: 20 (closest first)
    UBERON:0000989  penis
    UBERON:0011374  prepuce
    UBERON:0005156  reproductive structure
    UBERON:0034929  external soft tissue zone
    UBERON:0003133  reproductive organ
    UBERON:0004120  mesoderm-derived structure
    UBERON:0008811  intromittent

In [11]:
# Manually crafted lists and maps
# ---------------- 2) manual seeds and rule lists ----------------
def uid(label):
    """UBERON id for an exact label or EXACT synonym (fails loudly if missing or ambiguous)."""
    ids = lookup["label"].get(label) or lookup["exact"].get(label)
    assert ids and len(ids) == 1, f"{label!r}: {ids}"
    return next(iter(ids))

BLOOD, SKIN, HEMATO = uid("blood"), uid("skin of body"), uid("hematopoietic system")

# explicit string -> UBERON decisions (checked against every cleaned variant of a string)
MANUAL = {
    # blood and blood fractions
    "blood": BLOOD, "peripheral blood": BLOOD, "venous blood": BLOOD, "buffy coat": BLOOD,
    "mobilised peripheral blood": BLOOD, "mobilized peripheral blood": BLOOD,
    "pbmc": BLOOD, "pbmcs": BLOOD, "peripheral blood mononuclear cells": BLOOD,
    "peripheral blood mononuclear cell": BLOOD, "pbmcs from mtb exposed patients": BLOOD,
    "plasma": uid("blood plasma"), "plasma, extracted cell-free rna": uid("blood plasma"),
    "plasma extracellular vesicles": uid("blood plasma"),
    "serum": uid("blood serum"), "serum ev": uid("blood serum"),
    "cord blood": uid("umbilical cord blood"), "umbilical cord blood": uid("umbilical cord blood"),
    "cbmc": uid("umbilical cord blood"), "cbmcs": uid("umbilical cord blood"),
    "tumor educated platelets from blood": BLOOD,
    # hematopoietic, source not stated
    "blood or bone marrow": HEMATO, "bm": uid("bone marrow"),
    # lymph nodes
    "ln": uid("lymph node"), "mesenteric ln": uid("mesenteric lymph node"),
    # skin
    "psoriasis lesion": SKIN, "psoriatic skin": SKIN, "lesional skin": SKIN,
    "foreskin": uid("prepuce of penis"), "juvenile foreskin": uid("prepuce of penis"),
    # nose and airways
    "nasal": uid("nasal cavity"), "nasal lavage": uid("nasal cavity"), "saline nasal wash": uid("nasal cavity"),
    "nasal mucosa": uid("nasal cavity mucosa"),
    "bronchial epithelial cells": uid("epithelium of bronchus"), "bronchial epithelium": uid("epithelium of bronchus"),
    "brushing of bronchial epithelium": uid("epithelium of bronchus"),
    "cytologically normal bronchial airway epithelial cells": uid("epithelium of bronchus"),
    "small airway epithelium": uid("epithelium of bronchiole"),
    "small airway epithelium brushing": uid("epithelium of bronchiole"),
    "airway epithelium": uid("respiratory tract epithelium"), "airway epithelial cells": uid("respiratory tract epithelium"),
    "human airway epithelial cells": uid("respiratory tract epithelium"),
    "bronchoalveolar lavage": uid("lung"), "bronchoalveolar lavage (bal)": uid("lung"), "bal": uid("lung"),
    "sftpc+ cells": uid("lung"),                                       # alveolar type 2 cells
    "alveolar macrophages": uid("lung"),                               # from bronchoalveolar lavage
    # nervous system
    "hippocampus": uid("hippocampal formation"),
    "brain cortex": uid("cerebral cortex"), "cortex frontal": uid("frontal cortex"),
    "cortex motor lateral": uid("primary motor cortex"), "cortex motor medial": uid("primary motor cortex"),
    "cortex temporal": uid("temporal cortex"), "cortex occipital": uid("occipital cortex"),
    "ba9": uid("dorsolateral prefrontal cortex"),                     # Brodmann area 9
    "caudate": uid("caudate nucleus"), "ventral midbrain": uid("midbrain"),
    "meningioma": uid("meninx"), "retina punch": uid("retina"),
    "cns": uid("central nervous system"), "csf": uid("cerebrospinal fluid"),
    # digestive system
    "gut": uid("digestive tract"), "intestinal biopsy": uid("intestine"),
    "colon & rectum": uid("large intestine"), "crc": uid("large intestine"), "colorectal": uid("large intestine"),
    "oral mucosa": uid("mouth mucosa"),
    "islet": uid("islet of langerhans"), "islets": uid("islet of langerhans"),
    "islet cell": uid("islet of langerhans"), "islet cells": uid("islet of langerhans"),
    # reproductive system, placenta, embryo
    "endocervical cells": uid("endocervix"),
    "chorionic villous": uid("chorionic villus"), "chorionic villi": uid("chorionic villus"),
    "extravillous trophoblast": uid("placenta"), "preimplantation embryo": uid("embryo"),
    # heart, joints, fat
    "right atrial appendage": uid("right atrium auricular region"),
    "right atrial appendage biopsies": uid("right atrium auricular region"),
    "synovium": uid("synovium"), "synovial biopsy": uid("synovium"), "synovial tissue": uid("synovium"),
    "bulk synovial tissue": uid("synovium"),
    "visceral adipose tissue": uid("visceral abdominal adipose tissue"),
    # diseases that imply a site (not tumors)
    "ulcerative colitis": uid("large intestine"),
    "crohns disease": uid("intestine"), "crohn's disease": uid("intestine"),
}

# tumor names that imply an organ (tagged +tumor); also matched as single words inside longer strings
TUMOR_ORGAN = {
    "glioblastoma": uid("brain"), "gbm": uid("brain"), "glioma": uid("brain"), "astrocytoma": uid("brain"),
    "medulloblastoma": uid("cerebellum"),
    "pdac": uid("pancreas"),
    "hcc": uid("liver"), "hepatoblastoma": uid("liver"), "hepatocarcinoma": uid("liver"),
    "nsclc": uid("lung"), "luad": uid("lung"), "lusc": uid("lung"),
    "ccrcc": uid("kidney"), "rcc": uid("kidney"),
    "nmibc": uid("urinary bladder"), "mibc": uid("urinary bladder"),
    "npc tumor": uid("nasopharynx"),                                  # nasopharyngeal carcinoma ("npc" alone = neural progenitors)
    "primarypdac": uid("pancreas"),
    "aml": HEMATO, "cll": BLOOD,
}

# strings that never identify a tissue on their own: resolve later from other fields
NOT_A_TISSUE = {
    # tumors without an organ
    "tumor", "tumour", "tumors", "cancer", "primary tumor", "resected tumors", "tumor tissue",
    "baseline tumor tissue", "tumor cells", "tumor containing tissue", "lesion", "crpc metastasis",
    "human melanoma cells", "melanoma cells", "malignant ascites of agc", "dlbcl", "gep-net",
    "hnsc single-cell", "head and neck squamous cell carinoma (hnscc)", "head and squamous cell carcinoma",
    "single acute myeloid leukemia intitiating cell", "tumor infiltrating lymphocytes (tils)", "candidate ctc",
    "head and neck squamous cell carcinoma", "head & neck", "lscc tissue", "leukemia cells", "primary tumor ffpe",
    "cuimc meningioma patient tumor sequencing project",
    # cell types without a site
    "macrophages", "macrophage", "primary macrophages", "microglia", "endocrine cells", "endothelial cells",
    "chondrocytes", "cardiomyocytes", "astrocytes", "keratinocytes", "neuron", "neurons", "neuronal progenitors",
    "primordial germ cells", "oocyte", "ejaculated spermatozoa", "stem cells", "stem cell", "epcam+ cells",
    "cpm-high cells", "cell", "cells", "interneurons", "immune cell", "erythroid cells", "hepatocytes",
    "human retinal pigment epithelial cells", "rna from sorted cells",
    # generic words and leftovers
    "normal", "control", "sample", "biopsy", "biopsies", "periphery", "cell line", "human post mortem tissue",
    "transcriptome of single cells", "cdna", "neuronal epithelium", "inlet cells", "dahoun et al 2008",
    "hcs", "wbs", "ensc", "ls", "hn137p", "phlo", "hb", "nac", "tt", "nl",
    "adjacent normal", "adjacent normal mucosa tissue", "surgical specimen", "pre-treatment endoscopic biopsies",
    "knee/hip",
}

CONTROL_RNA = re.compile(r"reference rna|\buhrr\b|\bhbrr\b|\bmaqc\b|\bseqc\b|\berccs?\b")
XENOGRAFT = re.compile(r"\bpdx\b|xenograft|graft in (rat|mouse|mice)|engrafted|\bnsg\b|humani[sz]ed mice?")
CELL_LINE_NAMES = re.compile(
    r"\b(u87|a375|jurkat|huvecs?|mcf-?7|hela|hek-?293t?|k562|hepg2|a549|lncap|thp-?1|sh-?sy5y|hct-?116|caco-?2|"
    r"mda-?mb-?231|ht-?29|panc-?1|u2os|imr-?90|bj|rpe-?1|h295r|wm\d+\w*|coriell|atcc|lcls?|lymphoblastoid)\b|"
    r"umbilical vein endothelial|\bcell lines?\b")
IN_VITRO = re.compile(
    r"\bips\b|ipsc|induced pluripotent|embryonic stem|\bescs?\b|\bhescs?\b|\bh?pscs?\b|\bh[19]\b|"
    r"(pluripotent|neural|mesenchymal|embryonic) stem cells?|\bmscs?\b|"
    r"embryoid|organoid|blastoid|spheroid|stem[- ]cell[- ]derived|monocyte[- ]derived|psc[- ]derived|"
    r"derived from h?ipsc|progenitor cells|differentiat|\binduced (neurons?|astrocytes?|\w+ cells?)|-induced|\bicell\b|"
    r"cell culture|cultured|passage|fibroblasts?\b|myotubes?|myoblasts?|engineered|programmed|aortic endothelial|"
    r"^primary .*\b(cells?|hepatocytes|keratinocytes|myotubes)\b|^dcx[+-] d\d+$")
HEMATO_RE = re.compile(r"cd34\+|hematopoietic (stem |progenitor )?(and progenitor )?cells?|\bhspcs?\b|\bhscs?\b")
NASAL_EPI = re.compile(r"nasal (brush|brushing|epitheli|airway)|brushing of nasal")
NASOPHARYNX = re.compile(r"naso-?pharyn|oro-?naso-?pharyn")
# blood rule: a blood word or blood-cell word, and no other site
BLOOD_WORD = re.compile(r"\b(whole[- ]?)?blood\b")
BLOOD_CELL_WORD = re.compile(
    r"\b(t|b|nk|nkt)[- .]?cells?\b|\btcells?\b|\bbcells?\b|\btregs?\b|\bpbmcs?\b|"
    r"monocytes?|neutrophils?|thrombocytes?|platelets?|erythrocytes?|granulocytes?|lymphocytes?|leukocytes?|"
    r"eosinophils?|basophils?|plasma ?cells?|plasmablasts?|dendritic cells?|"
    r"^(cd\d+[+-]?|ccr\d+[+-]?)+$")                                  # bare marker strings: cd4, cd3+cd4+ccr7+
NON_BLOOD_SITE = re.compile(r"tumou?r|infiltrat|tissue|biops|lavage|synovi|skin|lung|liver|gut|intestin|colon|"
                            r"brain|csf|marrow|tonsil|lymph node|spleen|thymus|decidua|placenta|derived|culture|"
                            r"cord|menstrual")
TUMOR_WORDS = re.compile(r"\b(tumou?rs?|cancers?|carcinomas?|carinomas?|adenocarcinomas?|neoplasms?|"
                         r"metastas[ie]s|metastatic|malignant|primary|resected|baseline)\b")

# last resort: an organ word anywhere in the string (multi-word names first); ambiguous if two organs appear
ORGAN_WORDS = {w: uid(lbl) for w, lbl in [
    ("bone marrow", "bone marrow"), ("spinal cord", "spinal cord"), ("lymph node", "lymph node"),
    ("adipose", "adipose tissue"), ("small intestine", "small intestine"), ("liver", "liver"), ("lung", "lung"),
    ("kidney", "kidney"), ("heart", "heart"), ("brain", "brain"), ("cerebellum", "cerebellum"), ("colon", "colon"),
    ("stomach", "stomach"), ("pancreas", "pancreas"), ("spleen", "spleen"), ("thymus", "thymus"), ("skin", "skin of body"),
    ("breast", "breast"), ("prostate", "prostate gland"), ("ovary", "ovary"), ("testis", "testis"), ("uterus", "uterus"),
    ("placenta", "placenta"), ("retina", "retina"), ("esophagus", "esophagus"), ("bladder", "urinary bladder"),
    ("thyroid", "thyroid gland"), ("adrenal", "adrenal gland"), ("tonsil", "tonsil"), ("rectum", "rectum"),
    ("ileum", "ileum"), ("duodenum", "duodenum"), ("endometrium", "endometrium"), ("trachea", "trachea")]}
ORGAN_RE = re.compile(r"\b(" + "|".join(sorted(map(re.escape, ORGAN_WORDS), key=len, reverse=True)) + r")\b")

# text repair (the text is already lowercased, so broken UTF-8 is fixed by direct replacement)
MOJIBAKE = {"ã¯": "i", "ã©": "e", "ã¨": "e", "ã¶": "o", "ã¼": "u", "ã¤": "a", "ã±": "n", "â€“": "-", "â€™": "'"}
TYPOS = {"tisue": "tissue", "tisssue": "tissue", "periperal": "peripheral", "perpheral": "peripheral",
         "umblical": "umbilical", "oronasopharingeal": "oronasopharyngeal",
         "lymphoycyte": "lymphocyte", "lymphoycytes": "lymphocytes"}
ADJ = {"rectal": "rectum", "ileal": "ileum", "colonic": "colon", "gastric": "stomach", "hepatic": "liver",
       "renal": "kidney", "pulmonary": "lung", "cardiac": "heart", "duodenal": "duodenum", "jejunal": "jejunum",
       "esophageal": "esophagus", "oesophageal": "esophagus", "pancreatic": "pancreas", "splenic": "spleen",
       "thymic": "thymus", "dermal": "dermis", "cutaneous": "skin", "gingival": "gingiva",
       "endometrial": "endometrium", "placental": "placenta", "tonsillar": "tonsil", "bronchial": "bronchus",
       "tracheal": "trachea", "ovarian": "ovary", "testicular": "testis", "prostatic": "prostate",
       "endocervical": "endocervix", "cervical": "cervix", "mammary": "breast",
       "colorectal": "large intestine", "hepatocellular": "liver", "synovial": "synovium"}

In [12]:
# ---------------- 3) matcher ----------------
PREFIX = re.compile(r"^(human|donor \d+|donor|normal|healthy|adult|fetal|foetal|embryonic|primary|fresh|frozen|"
                    r"cryopreserved|total|whole|ffpe|post-?mortem|paired)\s+")
SUFFIX = re.compile(r"\s+(tissue|tissues|sample|samples|biopsy|biopsies|specimen|specimens|punch)$")
CELLS  = re.compile(r"\s+(cells?|cell population)$")


# Variants of a given string are generated using the following mechanisms:
# 1) Identity: original candidate is always included
# 2) PREFFIX and SUFFIX regexes are applied iteratively up to 3 times
# 3) Tracked adjectives are subsituted to nouns
# 4) Plurals are eliminated
# 5) CELL regexes are removed
# 6) "tissue" and "organ" are appended 
def _variants(s):
    out, seen = [], set()
    def add(x):
        x = x.strip(" ,;-")
        if x and x not in seen:
            seen.add(x); out.append(x)
    add(s); x = s
    for _ in range(3):
        x = SUFFIX.sub("", PREFIX.sub("", x)); add(x)
    for v in list(out):                                   # adjective -> noun ("rectal biopsy" -> "rectum")
        w = v.split(" ", 1)
        if w[0] in ADJ:
            add(SUFFIX.sub("", " ".join([ADJ[w[0]]] + w[1:])))
    for v in list(out):
        if v.endswith("s") and not v.endswith("ss"):
            add(v[:-1])
    for v in list(out):                                   # weaker variants last
        add(CELLS.sub("", v)); add(v + " tissue"); add(v + " organ")
    return out

# Additional variants are generated by removing parenthesis and tumor-related words
def variants(s):
    """Variants of s, then of s without (parentheses) / "week N", then without tumor words."""
    out = _variants(s)
    no_paren = re.sub(r"\s*\(.*?\)", "", s).strip()
    no_paren = re.sub(r"\s+(of\s+)?(gestational\s+)?weeks?\s+\d+", "", no_paren).strip()   # "... of week 24"
    no_tumor = re.sub(r"\s+", " ", TUMOR_WORDS.sub("", no_paren)).strip()
    for extra in (no_paren, no_tumor):
        if extra and extra != s:
            out += [v for v in _variants(extra) if v not in out]
    return out

def fix_text(s):
    for bad, good in MOJIBAKE.items():
        s = s.replace(bad, good)
    s = s.replace("ï", "i").replace("é", "e")
    return " ".join(TYPOS.get(w, w) for w in s.split())

def _hit(tid, how):
    return tid, terms[tid]["name"], how

# Function to match a string to an UBERON ID
def match(s):
    s = fix_text(s)
    vs = variants(s)
    tumor = "+tumor" if TUMOR_WORDS.search(s) else ""
    for v in vs:                                          # explicit decisions first, on every variant
        if v in MANUAL:
            return _hit(MANUAL[v], "manual")
    for v in vs:
        if v in TUMOR_ORGAN:
            return _hit(TUMOR_ORGAN[v], "tumor name+tumor")
    if s in NOT_A_TISSUE:
        return None, None, "not a tissue"
    if CONTROL_RNA.search(s):
        return None, None, "control rna"
    if XENOGRAFT.search(s):
        return None, None, "xenograft"
    if CELL_LINE_NAMES.search(s):
        return None, None, "cell line name"
    if IN_VITRO.search(s):
        return None, None, "in vitro"
    if HEMATO_RE.search(s):
        return _hit(HEMATO, "hematopoietic rule")
    if NASAL_EPI.search(s):
        return _hit(uid("nasal cavity epithelium"), "nasal rule")
    if NASOPHARYNX.search(s):
        return _hit(uid("nasopharynx"), "nasal rule")
    if (BLOOD_WORD.search(s) or BLOOD_CELL_WORD.search(s)) and not NON_BLOOD_SITE.search(s):
        return _hit(BLOOD, "blood rule")
    for w in re.findall(r"[a-z0-9]+", s):               # tumor abbreviations inside longer strings
        if w in TUMOR_ORGAN:
            return _hit(TUMOR_ORGAN[w], "tumor name+tumor")
    for kinds in (("label", "exact"), ("related",)):    # strong evidence on all variants first
        for v in vs:
            for kind in kinds:
                ids = lookup[kind].get(v)
                if ids:
                    if len(ids) > 1:
                        return None, None, f"ambiguous {kind}"
                    return _hit(next(iter(ids)), kind + ("" if v == s else "+variant") + tumor)
    organs = {ORGAN_WORDS[m] for m in ORGAN_RE.findall(" ".join(vs[:1] + [ADJ.get(w, w) for w in s.split()]))}
    if len(organs) == 1:
        return _hit(organs.pop(), "organ word" + tumor)
    if len(organs) > 1:
        return None, None, "ambiguous organ words"
    return None, None, "no match"

# ---------------- 4) match every string in the combined queue ----------------
cur = queues["combined"].copy()
cur[["uberon_id", "uberon_label", "match"]] = pd.DataFrame(cur["string"].map(match).tolist(), index=cur.index)
cur["category"] = np.select(
    [(cur["frac_cell_line"] >= 0.5) | (cur["match"] == "cell line name"),
     cur["match"] == "in vitro",
     cur["match"] == "xenograft",
     cur["match"] == "control rna",
     cur["match"] == "not a tissue",
     cur["uberon_id"].notna()],
    ["cell_line", "in_vitro", "xenograft", "control_rna", "needs_context", "tissue"], default="unmatched")
cur["tumor"] = cur["match"].str.endswith("+tumor")
cur["needs_review"] = (cur["match"].str.startswith(("related", "ambiguous"))
                       | (cur["category"] == "unmatched"))

# ---------------- 5) summary ----------------
n_all = len(samples)
by_cat = cur.groupby("category")["n_samples"].sum().sort_values(ascending=False)
print("samples by category (all strings):")
print((by_cat / n_all).map("{:.1%}".format).to_string())
print("\nmatch type among matched strings (samples):")
print((cur[cur["uberon_id"].notna()].groupby("match")["n_samples"].sum() / n_all).map("{:.1%}".format).to_string())

top = cur.head(N_REVIEW)
print(f"\ntop {N_REVIEW} strings: {top['uberon_id'].notna().mean():.1%} auto-mapped, "
      f"{top['needs_review'].sum():,} need review, covering {top['n_samples'].sum() / n_all:.1%} of samples")

print("\nmost frequent strings still needing a decision (needs_context excluded):")
print(top.loc[top["needs_review"],
              ["field", "string", "n_samples", "n_series", "frac_cell_line", "match"]].head(40).to_string())

# ---------------- 6) attach the automatic labels to the samples ----------------
key = pd.MultiIndex.from_arrays([
    np.where(samples["tissue_value"].notna(), "tissue", "source_name"),
    samples["tissue_value"].fillna(samples["source_name"])])
lab = cur.set_index(["field", "string"])
samples["uberon_auto"]   = lab["uberon_id"].reindex(key).to_numpy()
samples["uberon_match"]  = lab["match"].reindex(key).to_numpy()
samples["category_auto"] = lab["category"].reindex(key).to_numpy()
samples["tumor_auto"]    = lab["tumor"].reindex(key, fill_value=False).to_numpy()
print(f"\nsamples with an automatic UBERON id: {samples['uberon_auto'].notna().mean():.1%}")

samples by category (all strings):
category
tissue           46.0%
cell_line        32.1%
in_vitro          8.4%
unmatched         7.9%
needs_context     4.6%
xenograft         0.8%
control_rna       0.3%

match type among matched strings (samples):
match
blood rule                5.7%
exact                     2.9%
exact+variant             1.9%
exact+variant+tumor       0.5%
hematopoietic rule        0.7%
label                    10.4%
label+tumor               0.0%
label+variant             4.6%
label+variant+tumor       2.4%
manual                   16.0%
nasal rule                0.6%
organ word                2.3%
organ word+tumor          1.1%
related                   0.1%
related+variant           0.1%
related+variant+tumor     0.0%
tumor name+tumor          1.0%

top 2000 strings: 44.0% auto-mapped, 234 need review, covering 72.5% of samples

most frequent strings still needing a decision (needs_context excluded):
            field                                        strin

### 3) Review mapping

In [13]:
review = (samples[samples["uberon_auto"].notna()]
          .groupby("uberon_match", group_keys=False)
          .apply(lambda d: d.sample(min(len(d), 20), random_state=0)))
review["uberon_label"] = review["uberon_auto"].map(lambda t: terms[t]["name"])
review[["uberon_match", "uberon_label", "tissue_value", "source_name", "title"]]
# review

/scratch/slurm-job.9966077/ipykernel_113450/3201527996.py:3: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda d: d.sample(min(len(d), 20), random_state=0)))


,uberon_match,uberon_label,tissue_value,source_name,title
sample_id,,,,,
GSM3717098,blood rule,blood,NaN,b cell clones,20
GSM5919268,blood rule,blood,platelet,platelet,tic92
GSM2842514,blood rule,blood,NaN,subject21 visit 1 pbmc il7rhigh temra,bulk rna-seq subject21 v1 il7rhigh temra
GSM5540590,blood rule,blood,NaN,islet-reactive cd4+ sorted t-cell,tid12 a7 t1d t1d12 lib29445 ccfmdanxx tophat a...
GSM4844574,blood rule,blood,NaN,eosinophils,c-4h-resting-rep2
...,...,...,...,...,...
GSM2037149,tumor name+tumor,hematopoietic system,NaN,aml 51 (7934),51 epz004777
GSM4658332,tumor name+tumor,brain,NaN,diffuse intrinsic pontine glioma,dipg4 ptc028 3
GSM4552747,tumor name+tumor,blood,NaN,primary cll cells,p7 notch.replicate1


### 4) Map remaining with low confidence

1) Use remaining GEO fields
2) Expression classifier
3) Text2term fuzzy mapping

In [20]:
from cellxgene_ontology_guide.ontology_parser import OntologyParser
from cellxgene_ontology_guide.curated_ontology_term_lists import get_curated_ontology_term_list
from cellxgene_ontology_guide.entities import CuratedOntologyTermList

parser = OntologyParser()                       # or OntologyParser(schema_version="v<X.Y.Z>") to match your Census release
TG = get_curated_ontology_term_list(CuratedOntologyTermList.TISSUE_GENERAL)

def to_tissue_general(tid):
    try:
        hits = parser.get_high_level_terms(tid, TG)
        return hits[0] if hits else None
    except (KeyError, ValueError, IndexError):  # id unknown to CELLxGENE's UBERON version
        return None

In [21]:
# =====================================================================================
# A) sample-level cell-line override, then other text fields (cell type, then title)
# =====================================================================================
TODO_CATS = {"unmatched", "needs_context"}
EXCLUDE = {"cell line name": "cell_line", "in vitro": "in_vitro", "xenograft": "xenograft", "control rna": "control_rna"}
_tg_cache = {}
def roll_up(tid):
    if tid not in _tg_cache:
        _tg_cache[tid] = to_tissue_general(tid)
    return _tg_cache[tid]

samples["label_source"] = pd.Series(np.where(samples["category_auto"] == "tissue", "text", None), index=samples.index, dtype=object)
cl = samples["has_cell_line"] & (samples["category_auto"] != "cell_line")
samples.loc[cl, "category_auto"] = "cell_line"                                 # sample-level cell-line flag wins
samples.loc[cl, "label_source"] = None
print(f"cell-line override: {cl.sum():,} samples moved to cell_line")

for col in ["cell_type_value", "title"]:
    todo = samples["category_auto"].isin(TODO_CATS) & samples[col].notna()
    res = {s: match(s) for s in samples.loc[todo, col].unique()}
    tid = samples.loc[todo, col].map(lambda s: res[s][0])
    how = samples.loc[todo, col].map(lambda s: res[s][2])
    hit = tid.notna() & ~how.str.startswith("related")                        # no weak synonyms from secondary fields
    i = hit[hit].index
    samples.loc[i, "uberon_auto"] = tid[i]
    samples.loc[i, "category_auto"] = "tissue"
    samples.loc[i, "label_source"] = col
    samples.loc[i, "uberon_match"] = col + ":" + how[i]
    ex = how[~hit].map(EXCLUDE).dropna()
    samples.loc[ex.index, "category_auto"] = ex
    print(f"{col}: {len(i):,} samples got a tissue, {len(ex):,} moved to an excluded category")

new = samples["label_source"].isin(["cell_type_value", "title"])
samples.loc[new, "tissue_general_id"] = samples.loc[new, "uberon_auto"].map(roll_up)

cell-line override: 25,589 samples moved to cell_line
cell_type_value: 9,939 samples got a tissue, 9,448 moved to an excluded category
title: 5,967 samples got a tissue, 1,380 moved to an excluded category


In [22]:
# =====================================================================================
# B) series majority vote on tissue_general
# =====================================================================================
MIN_VOTERS, MIN_SHARE = 3, 0.8
series = samples["series_id"].astype(str).str.split(",").str[0]
voters = samples["label_source"].isin(["text", "cell_type_value", "title"]) & samples["tissue_general_id"].notna()
v = pd.DataFrame({"series": series[voters], "tg": samples.loc[voters, "tissue_general_id"]})
best = (v.groupby(["series", "tg"]).size().rename("n").reset_index()
        .sort_values("n", ascending=False).drop_duplicates("series").set_index("series"))   # top tissue per series
best["n_total"] = v.groupby("series").size()
good = best.loc[(best["n_total"] >= MIN_VOTERS) & (best["n"] / best["n_total"] >= MIN_SHARE), "tg"]
todo = samples["category_auto"].isin(TODO_CATS) & series.isin(good.index)
samples.loc[todo, "tissue_general_id"] = series[todo].map(good)
samples.loc[todo, "uberon_auto"] = samples.loc[todo, "tissue_general_id"]     # only the coarse level is known
samples.loc[todo, "category_auto"] = "tissue"
samples.loc[todo, "label_source"] = "series_vote"
samples.loc[todo, "uberon_match"] = "series vote"
print(f"series vote: {todo.sum():,} samples labelled from {series[todo].nunique():,} series "
      f"(>= {MIN_VOTERS} labelled samples, >= {MIN_SHARE:.0%} agreement)")

series vote: 1,502 samples labelled from 55 series (>= 3 labelled samples, >= 80% agreement)


In [ ]:
# =====================================================================================
# C) expression classifier: train on text-labelled samples, predict the rest
# =====================================================================================
import h5py
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import accuracy_score, f1_score

EXPR_KEY, BLOCK, SEED = "expression", 2000, 0
MIN_CLASS, PER_CLASS, N_HVG, N_PCS, HVG_ROWS, MIN_PROB = 300, 1500, 5000, 100, 8000, 0.8
rng = np.random.default_rng(SEED)
pos = pd.Series(np.arange(len(metadata)), index=metadata["sample_id"].astype(str))   # h5 rows follow metadata.csv

def read_logcpm(rows, genes=None):
    """log1p(CPM) for h5 row positions `rows` (any order), optionally only `genes` columns."""
    assert len(rows), "no rows to read"
    order = np.argsort(rows)
    parts = []
    with h5py.File(EXPRESSION_H5, "r") as f:
        ds = f[EXPR_KEY]
        for s in range(0, len(order), BLOCK):
            X = ds[rows[order[s:s + BLOCK]], :].astype(np.float32)             # increasing positions for h5py
            lib = X.sum(1, keepdims=True); lib[lib == 0] = 1
            X = np.log1p(X / lib * 1e6)
            parts.append(X if genes is None else X[:, genes])
    out = np.empty((len(rows), parts[0].shape[1]), dtype=np.float32)
    out[order] = np.vstack(parts)
    return out

# training set: text-labelled tissue samples of well-populated classes, balanced, capped per class
lab = samples[samples["label_source"].isin(["text", "cell_type_value", "title"])
              & samples["tissue_general_id"].notna()]
counts = lab["tissue_general_id"].value_counts()
classes = counts[counts >= MIN_CLASS].index
train = (lab[lab["tissue_general_id"].isin(classes)]
         .sample(frac=1, random_state=SEED)                                    # shuffle, then cap per class
         .groupby("tissue_general_id").head(PER_CLASS))
print(f"training: {len(train):,} samples, {len(classes)} classes (>= {MIN_CLASS} labelled samples each)")

hvg_rows = pos[rng.choice(train.index, min(HVG_ROWS, len(train)), replace=False)].to_numpy()
genes = np.argsort(read_logcpm(hvg_rows).var(0))[::-1][:N_HVG]               # most variable genes
X = read_logcpm(pos[train.index].to_numpy(), genes)
y = train["tissue_general_id"].to_numpy()
g = series[train.index].to_numpy()

def fit(Xa, ya):
    pca = PCA(N_PCS, random_state=SEED).fit(Xa)
    clf = LogisticRegression(max_iter=2000, C=1.0).fit(pca.transform(Xa), ya)
    return pca, clf

tr, te = next(GroupShuffleSplit(1, test_size=0.2, random_state=SEED).split(X, y, g))   # held-out series
pca, clf = fit(X[tr], y[tr])
p = clf.predict_proba(pca.transform(X[te])); pred = clf.classes_[p.argmax(1)]; conf = p.max(1) >= MIN_PROB
print(f"held-out series: accuracy {accuracy_score(y[te], pred):.3f}, macro-F1 {f1_score(y[te], pred, average='macro'):.3f}; "
      f"at prob >= {MIN_PROB}: {conf.mean():.1%} kept, accuracy {accuracy_score(y[te][conf], pred[conf]):.3f}")

pca, clf = fit(X, y)                                                          # final model on all training data
todo = samples.index[samples["category_auto"].isin(TODO_CATS)]
assert len(todo), "nothing left to predict"
p = clf.predict_proba(pca.transform(read_logcpm(pos[todo].to_numpy(), genes)))
samples.loc[todo, "expr_pred"] = clf.classes_[p.argmax(1)]
samples.loc[todo, "expr_prob"] = p.max(1)
ok = todo[p.max(1) >= MIN_PROB]
samples.loc[ok, "tissue_general_id"] = samples.loc[ok, "expr_pred"]
samples.loc[ok, "uberon_auto"] = samples.loc[ok, "expr_pred"]
samples.loc[ok, "category_auto"] = "tissue"
samples.loc[ok, "label_source"] = "expression"
samples.loc[ok, "uberon_match"] = "expression"
print(f"expression: {len(ok):,} of {len(todo):,} remaining samples labelled (prob >= {MIN_PROB})")

samples["tissue_general"] = samples["tissue_general_id"].map(
    lambda t: parser.get_term_label(t) if isinstance(t, str) else None)

training: 7,847 samples, 9 classes (>= 300 labelled samples each)
held-out series: accuracy 0.468, macro-F1 0.317; at prob >= 0.8: 86.6% kept, accuracy 0.492


/cluster/apps/biomed/boeva/rquiles/conda/envs/bulkFM/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [ ]:
# =====================================================================================
# D) text2term cross-check of the inferred labels (series vote and expression)
# =====================================================================================
import text2term
UBERON_OWL = "/cluster/work/boeva/rquiles/ontologies/uberon-basic.owl"
if not text2term.cache_exists("UBERON"):
    text2term.cache_ontology(UBERON_OWL, "UBERON", base_iris=("http://purl.obolibrary.org/obo/UBERON",))

inferred = samples["label_source"].isin(["series_vote", "expression"])
prim = samples["tissue_value"].fillna(samples["source_name"])
strings = prim[inferred].dropna().unique().tolist()
t2t = text2term.map_terms(strings, "UBERON", use_cache=True, min_score=0.6, max_mappings=1)
t2t_tg = dict(zip(t2t["Source Term"], t2t["Mapped Term CURIE"].map(roll_up))) if len(t2t) else {}
samples.loc[inferred, "t2t_tissue_general_id"] = prim[inferred].map(t2t_tg)

chk = samples[inferred & samples["t2t_tissue_general_id"].notna()]
agree = (chk["t2t_tissue_general_id"] == chk["tissue_general_id"])
print(f"text2term proposals for {len(chk):,} of {inferred.sum():,} inferred samples; "
      f"agreement with the inferred label: {agree.mean() if len(chk) else float('nan'):.1%}")
print(chk.assign(agree=agree).groupby("label_source")["agree"].mean().map("{:.1%}".format).to_string())

# =====================================================================================
# summary
# =====================================================================================
print("\nlabel source (all samples):")
print(samples["label_source"].fillna("none").value_counts(normalize=True).map("{:.1%}".format).to_string())
print("\ncategory (all samples):")
print(samples["category_auto"].value_counts(normalize=True).map("{:.1%}".format).to_string())

## UBERON IDs --> CellxGene Categories

### Map to CellxGene Categories

In [47]:
tg = {t: to_tissue_general(t) for t in samples["uberon_auto"].dropna().unique()}
samples["tissue_general_id"] = samples["uberon_auto"].map(tg)
samples["tissue_general"] = samples["tissue_general_id"].map(lambda t: parser.get_term_label(t) if isinstance(t, str) else None)

unmapped = [t for t, v in tg.items() if v is None]
print(f"{len(tg)} UBERON ids -> {len(set(tg.values()) - {None})} tissue_general categories; {len(unmapped)} ids without one: "
      + ", ".join(f"{t} ({terms[t]['name']})" for t in unmapped[:10]))
print(samples.loc[samples["category_auto"] == "tissue", "tissue_general"].value_counts(dropna=False).head(40).to_string())

print(f"all samples:      {samples['tissue_general_id'].notna().mean():.1%}")
print(f"tissue category:  {samples.loc[samples['category_auto'] == 'tissue', 'tissue_general_id'].notna().mean():.1%}")
print(samples.groupby("category_auto", dropna=False)["tissue_general_id"].apply(lambda s: f"{s.notna().mean():.1%}"))

552 UBERON ids -> 59 tissue_general categories; 59 ids without one: UBERON:0000106 (zygote stage), UBERON:0001851 (cortex), UBERON:0001968 (semen), UBERON:0000058 (duct), UBERON:0000173 (amniotic fluid), UBERON:0007376 (outer epithelium), UBERON:0001090 (synovial fluid), UBERON:0000479 (tissue), UBERON:0000483 (epithelium), UBERON:0004345 (trophectoderm)
tissue_general
blood                     135141
skin of body               18963
pancreas                   17061
brain                      16519
bone marrow                15090
liver                      13672
lung                       11238
breast                      7281
hematopoietic system        6288
colon                       5596
fallopian tube              5539
adipose tissue              5231
musculature                 4810
embryo                      4647
kidney                      4625
heart                       4603
None                        4372
nose                        4321
uterus                      4233
l

###